# 🎯 Module 09: LSTM & GRU

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** LSTM & GRU  
> **Framework:** PyTorch  
> **Date:** 09 June 2026  
> **Previous:** 08 — RNN  
> **Next:** 10 — Transformers Basics

---

In Module 08 we saw that a **vanilla RNN** struggles with **long-term dependencies** — gradients vanish or explode, and the hidden state gets overwritten at every step.

**LSTM** and **GRU** solve this by adding **gates** — small neural networks that decide what to keep, what to forget, and what to output at each time step.

In this notebook we will:

- Understand **why** gating is needed
- Walk through the **LSTM equations** step by step
- Understand the **cell state** vs the **hidden state**
- Cover **GRU**'s update and reset gates
- Implement **LSTM** and **GRU** models in PyTorch
- Build and train a full **LSTM classifier**
- Compare RNN vs LSTM vs GRU empirically

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain **why** LSTM / GRU were invented
- Describe the **cell state** and **hidden state** roles
- Write the **forget / input / output** gate equations
- Compute an LSTM step **by hand**
- Explain **GRU**'s update and reset gates
- Compare **LSTM vs GRU** structure & parameters
- Use `nn.LSTM` and `nn.GRU` correctly in PyTorch
- Handle `output, (h, c) = lstm(x)` vs `output, h = gru(x)`
- Build, train and evaluate an LSTM classifier
- Empirically compare **RNN vs LSTM vs GRU**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import math
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. Why LSTM and GRU?

Vanilla RNNs can remember previous information through their hidden state, but they struggle with **long-term dependencies**.

Example:

```text
I grew up in Pakistan. ...
[Many words later]
... I speak Urdu.
```

The model may need information from **much earlier** in the sequence.

During **BPTT**, gradients can become extremely small or extremely large:

- **Vanishing** gradients → no learning signal from distant steps
- **Exploding** gradients → unstable training, NaNs
- **Result** → difficulty learning long-term dependencies

### The fix: gates

LSTM and GRU use **gates** — small sigmoid-activated neural networks — that control information flow:

```text
0 → mostly reject
1 → mostly keep
```

This gives the network a **learned memory controller**.

---
## 🔒 2. LSTM — Long Short-Term Memory

An LSTM has **two states**:

```text
Hidden state → hₜ   (short-term, exposed to next layers)
Cell state   → cₜ   (long-term, internal memory highway)
```

The **cell state** is the key innovation: it flows along the sequence with only **element-wise** operations, which makes gradients much better behaved than in a vanilla RNN.

### Three gates

```text
             ┌─────────────┐
xₜ ─────────→│ Forget Gate │
             ├─────────────┤
xₜ ─────────→│ Input Gate  │
             ├─────────────┤
xₜ ─────────→│ Output Gate │
             └─────────────┘
                    ↓
               Cell State
```

The gates use **sigmoid** so their outputs are in $(0, 1)$:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

---
## 🧮 3. LSTM Equations

### Forget gate — what to remove

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$

### Input gate — what to add

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$

### Candidate cell state

$$\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$$

### Cell state update — the heart of LSTM

$$\boxed{c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t}$$

### Output gate — what to expose

$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$

### Hidden state

$$h_t = o_t \odot \tanh(c_t)$$

where $\odot$ is element-wise multiplication.

### Mental model

> **LSTM learns what to forget, what to remember, and what to output.**

---
## ✏️ 4. A Single LSTM Step by Hand

Given:

```text
old cell state = 0.8
forget gate    = 0.9
input gate     = 0.2
candidate      = 0.5
```

Then:

$$c_t = (0.9)(0.8) + (0.2)(0.5) = 0.72 + 0.10 = 0.82$$

The model **mostly retained** the previous memory while **adding a little** new information.

That's the key intuition: the **cell state evolves smoothly**, with the gates deciding *how much* of each contribution to keep.

In [ ]:
# Verify the LSTM step
c_prev, f, i, c_tilde = 0.8, 0.9, 0.2, 0.5
c_new = f * c_prev + i * c_tilde
print(f'c_{{t-1}} = {c_prev}')
print(f'f_t     = {f}  (mostly keep)')
print(f'i_t     = {i}  (little new info)')
print(f'c̃_t     = {c_tilde}')
print(f'c_t     = {c_new:.4f}')

In [ ]:
# Visualize sigmoid: the gate activation function
z = np.linspace(-8, 8, 400)
sig = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 3.5))
plt.plot(z, sig, label='sigmoid(z)')
plt.axhline(0, color='k', linewidth=0.5)
plt.axhline(1, color='k', linewidth=0.5, linestyle=':')
plt.title('Sigmoid — used in LSTM / GRU gates  (range: 0 → 1)')
plt.xlabel('z'); plt.ylabel('gate value')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🔓 5. GRU — Gated Recurrent Unit

GRU is a **simpler** gated recurrent architecture.

| | LSTM | GRU |
| --- | --- | --- |
| Hidden state | ✅ | ✅ |
| Cell state | ✅ | ❌ |
| Main gates | 3 (forget, input, output) | 2 (update, reset) |
| Parameters | More | Fewer |
| Complexity | Higher | Lower |

### Update gate — how much old info to keep

$$z_t = \sigma(W_z [x_t, h_{t-1}] + b_z)$$

### Reset gate — how much past to consider when proposing new info

$$r_t = \sigma(W_r [x_t, h_{t-1}] + b_r)$$

### Candidate hidden state

$$\tilde{h}_t = \tanh(W_h [x_t,\ r_t \odot h_{t-1}] + b_h)$$

### Final hidden state

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### Intuition

```text
Update gate → how much to refresh
Reset gate  → how much of the past to use when proposing new info
```

---
## 🐍 6. PyTorch LSTM & GRU

### LSTM

```python
lstm = nn.LSTM(input_size=50, hidden_size=64, batch_first=True)
output, (hidden, cell) = lstm(x)
```

### GRU

```python
gru = nn.GRU(input_size=50, hidden_size=64, batch_first=True)
output, hidden = gru(x)
```

### The key difference in the API

```text
LSTM → output, (hidden, cell)
GRU  → output, hidden
```

### Shapes (one layer, batch_first=True)

| Tensor | Shape | Meaning |
| --- | --- | --- |
| output | `[B, T, H]` | hidden at every time step |
| hidden | `[1, B, H]` | final hidden state |
| cell   | `[1, B, H]` | final cell state (LSTM only) |

In [ ]:
# Verify LSTM output shapes
B, T, F, H = 32, 20, 50, 64
x = torch.randn(B, T, F)

lstm = nn.LSTM(input_size=F, hidden_size=H, batch_first=True)
output, (hidden, cell) = lstm(x)
print('LSTM:')
print('  input  →', x.shape)
print('  output →', output.shape)
print('  hidden →', hidden.shape)
print('  cell   →', cell.shape)

In [ ]:
# Verify GRU output shapes
gru = nn.GRU(input_size=F, hidden_size=H, batch_first=True)
output, hidden = gru(x)
print('GRU:')
print('  input  →', x.shape)
print('  output →', output.shape)
print('  hidden →', hidden.shape)

In [ ]:
# Parameter count comparison: RNN vs LSTM vs GRU
def count(m):
    return sum(p.numel() for p in m.parameters())

F, H = 50, 64
rnn  = nn.RNN (input_size=F, hidden_size=H, batch_first=True)
lstm = nn.LSTM(input_size=F, hidden_size=H, batch_first=True)
gru  = nn.GRU (input_size=F, hidden_size=H, batch_first=True)

print(f'RNN  params: {count(rnn):>6}')
print(f'LSTM params: {count(lstm):>6}   (~4× RNN)')
print(f'GRU  params: {count(gru):>6}   (~3× RNN)')

---
## 🏛️ 7. Building an LSTM Classifier

```text
Sequence
   ↓
LSTM
   ↓
Final hidden state (hidden[-1])
   ↓
Linear
   ↓
Class logits
```

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, (hidden, cell) = self.lstm(x)
        final_hidden = hidden[-1]           # last layer's hidden state
        return self.fc(final_hidden)


class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, hidden = self.gru(x)
        return self.fc(hidden[-1])


class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.fc(hidden[-1])


# Shape check for all three
x_demo = torch.randn(4, 20, 8)
for name, model in [('RNN ', RNNClassifier(8, 64, 2)),
                    ('LSTM', LSTMClassifier(8, 64, 2)),
                    ('GRU ', GRUClassifier(8, 64, 2))]:
    y = model(x_demo)
    print(f'{name}  → input {tuple(x_demo.shape)}  output {tuple(y.shape)}  '
          f'params={sum(p.numel() for p in model.parameters())}')

---
## 🧪 8. Practical Experiment: RNN vs LSTM vs GRU

We'll train the **same task** with three architectures and compare:

- Training loss
- Validation loss
- Accuracy
- Parameter count
- Training time

We'll use a **longer sequence** (T=50) with a signal that requires remembering the **first few steps** — this exposes the vanilla RNN's weakness on long-range dependencies.

In [ ]:
# Synthetic long-range classification dataset
torch.manual_seed(42)
N_TRAIN, N_VAL = 2000, 500
T, F = 50, 4

def make_long_range_data(n):
    """Signal is encoded in the FIRST few steps, not the mean."""
    X = torch.randn(n, T, F)
    # Signal = the sign of feature 0's value at t=0..2
    signal = X[:, :3, 0].sum(dim=1)
    y = (signal > 0).long()
    return X, y

X_train, y_train = make_long_range_data(N_TRAIN)
X_val,   y_val   = make_long_range_data(N_VAL)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=128, shuffle=False)

print('Train:', X_train.shape, 'Val:', X_val.shape)
print('Positive fraction (train):', y_train.float().mean().item())
print('Note: the signal is at the START of a 50-step sequence — requires long memory.')

In [ ]:
# Train + eval helpers
def train_one_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        if clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

def run_experiment(name, model_cls, epochs=15):
    torch.manual_seed(42)
    model = model_cls(input_size=F, hidden_size=64, num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(va_loss)
        hist['train_acc'].append(tr_acc);   hist['val_acc'].append(va_acc)
    dt = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    print(f'{name:6s} | val_acc={hist["val_acc"][-1]:.3f} | '
          f'params={n_params:>6} | time={dt:.1f}s')
    return hist, n_params, dt

print('Training 3 models on a long-range task (T=50)…\n')
hist_rnn,  p_rnn,  t_rnn  = run_experiment('RNN ', RNNClassifier)
hist_lstm, p_lstm, t_lstm = run_experiment('LSTM', LSTMClassifier)
hist_gru,  p_gru,  t_gru  = run_experiment('GRU ', GRUClassifier)

In [ ]:
# Compare learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for hist, label in [(hist_rnn, 'RNN'), (hist_lstm, 'LSTM'), (hist_gru, 'GRU')]:
    axes[0].plot(hist['val_loss'], label=label)
    axes[1].plot(hist['val_acc'],  label=label)

axes[0].set_title('Validation loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('RNN vs LSTM vs GRU on a long-range sequence task (T=50)')
plt.tight_layout(); plt.show()

In [ ]:
# Parameter / time summary table
rows = [
    ('RNN ',  p_rnn,  t_rnn,  hist_rnn ['val_acc'][-1]),
    ('LSTM', p_lstm, t_lstm, hist_lstm['val_acc'][-1]),
    ('GRU ',  p_gru,  t_gru,  hist_gru ['val_acc'][-1]),
]

print(f'{"Model":<6} {"Params":>10} {"Time (s)":>10} {"Val Acc":>10}')
print('-' * 42)
for name, p, t, a in rows:
    print(f'{name:<6} {p:>10,} {t:>10.1f} {a:>10.3f}')

print('\nObservation:')
print('  - Vanilla RNN often plateaus low on long-range tasks.')
print('  - LSTM / GRU retain useful signal from early steps (better accuracy).')
print('  - GRU has fewer params than LSTM, often competitive accuracy.')

---
## 🎯 9. When to Use LSTM / GRU

Common applications:

- Text classification
- Sentiment analysis
- Time-series prediction
- Speech-related sequence tasks
- Sequence labeling (POS, NER)
- Sequential sensor data
- Event prediction

Modern Transformer architectures are often preferred for large-scale NLP, but **LSTM / GRU remain important**:

- For understanding sequence modeling
- In smaller or resource-constrained systems
- In streaming / online settings where state must be maintained
- When sequence lengths are modest and compute is limited

---
## 🚨 10. Common Mistakes

### ❌ 1. Confusing LSTM states
LSTM returns:
```python
output, (hidden, cell) = lstm(x)
```
Not `output, hidden`.

### ❌ 2. Forgetting `batch_first=True`
Without it, PyTorch expects `[T, B, F]`, not `[B, T, F]`.

### ❌ 3. Using future information
For forecasting, future data must **never** enter the input — that's data leakage.

### ❌ 4. Assuming GRU is always better
GRU has fewer parameters, but performance depends on the actual task and data.

### ❌ 5. Softmax before CrossEntropyLoss
Return **logits** and let `CrossEntropyLoss` handle the transformation:
```python
loss = nn.CrossEntropyLoss()(logits, y)      # ✅
```

### ❌ 6. Forgetting gradient clipping
Even LSTM / GRU can produce exploding gradients — always clip.
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
```

### ❌ 7. Forgetting `model.eval()` at inference
Dropout and other layers behave differently during evaluation.

---
## 🏋️ 11. Hands-On Exercises

### Exercise 1 — Hidden Size Sweep
Train LSTM with `hidden_size ∈ {32, 64, 128}`. Plot val accuracy vs hidden size.

### Exercise 2 — Stacked LSTM
Use `num_layers=2` or 3 with `dropout=0.2`. Does it help on this long-range task?

### Exercise 3 — Bidirectional LSTM
Use `bidirectional=True`. Adjust the classifier input size (hidden × 2). Compare accuracy and training time.

### Exercise 4 — LSTM without Forget Gate
Set `f_t = 1` (no forgetting). Retrain. Does performance drop?

### Exercise 5 — GRU vs LSTM Parameters
Compute parameter counts analytically for `input_size=F`, `hidden_size=H` for both. Verify with PyTorch.

### Exercise 6 — Time-Series Regression
Replace the classification head with `nn.Linear(hidden, 1)` and `nn.MSELoss()`. Predict the next value of `sin(t)` from the previous 20 steps.

### Exercise 7 — Attention over LSTM
Use `output` (all hidden states) and apply a simple **additive attention** over time to pool them. Does it beat `hidden[-1]`?

---
## 🎤 12. Interview Questions

1. **What problem does LSTM solve?**  
   It improves the ability of recurrent networks to learn **long-term dependencies**.

2. **What are the main LSTM gates?**  
   **Forget**, **input**, and **output** gates.

3. **What is the cell state?**  
   A memory pathway that carries information through the sequence.

4. **What is GRU?**  
   A gated recurrent architecture with a **simpler** structure than LSTM.

5. **LSTM vs GRU?**  
   LSTM has a separate cell state and more gates; GRU combines information into a single hidden state with fewer gates and fewer parameters.

6. **Why use sigmoid in gates?**  
   It produces values between 0 and 1, letting the network control information flow smoothly.

7. **What's the difference in PyTorch's API?**  
   `output, (hidden, cell) = lstm(x)` vs `output, hidden = gru(x)`.

8. **Why do LSTM / GRU help with vanishing gradients?**  
   The cell state provides an **additive** path for gradients — much better than the multiplicative path in a vanilla RNN.

9. **Do they eliminate the need for gradient clipping?**  
   No — exploding gradients can still happen; clipping is still recommended.

10. **When would you prefer GRU over LSTM?**  
    When you want fewer parameters and faster training, and the task doesn't need the full LSTM gating capacity.

---
## ✅ 13. Knowledge Check

- [ ] Explain why LSTM / GRU were invented
- [ ] Explain the cell state vs hidden state
- [ ] Write the forget gate equation
- [ ] Write the input gate equation
- [ ] Write the output gate equation
- [ ] Write the cell state update equation
- [ ] Compute a single LSTM step by hand
- [ ] Explain the update gate in GRU
- [ ] Explain the reset gate in GRU
- [ ] Compare LSTM vs GRU structure & params
- [ ] Use `nn.LSTM` and `nn.GRU` correctly
- [ ] Handle `(hidden, cell)` vs `hidden`
- [ ] Build and train an LSTM classifier
- [ ] Compare RNN / LSTM / GRU empirically

---
## 🏁 14. Key Takeaways

### The evolution

```text
Vanilla RNN
    ↓
Long-term dependency problems
    ↓
LSTM / GRU
    ↓
Gated information flow
```

### Architecture summary

```text
LSTM
├── Forget gate
├── Input gate
├── Cell state
├── Output gate
└── Hidden state

GRU
├── Update gate
├── Reset gate
└── Hidden state
```

### The core LSTM equation

$$\boxed{c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t}$$

### The main idea

> **RNNs provide memory; LSTM and GRU provide controlled memory.**

### 🧭 Golden Rule

> **LSTM and GRU don't eliminate gradient problems — but they make long-range learning practical. Always clip gradients and never let future information leak into a sequence model.**

---

### 🔗 What's Next?

**Module 10 — Transformers Basics**

LSTMs and GRUs still process sequences **step by step**. Transformers replace recurrence with **attention** — enabling **direct token-to-token interaction** and **highly parallel training**.

```text
RNN
 ↓
LSTM / GRU
 ↓
Attention
 ↓
Transformer
 ↓
Modern LLMs
```